# 🧪 Thực Hành: Tự Code Logistic Regression từ Con Số Không (Scratch)

Chào mừng Khang đến với bài thực hành Logistic Regression! Trong bài tập này, bạn sẽ tự tay lập trình từng bộ phận cấu thành nên mô hình Logistic Regression bằng Python và thư viện `numpy`. 

Việc tự xây dựng mô hình giúp bạn hiểu sâu sắc nguyên lý hoạt động của:
1. Hàm kích hoạt **Sigmoid**.
2. Hàm mất mát **Binary Cross-Entropy**.
3. Thuật toán tối ưu hóa **Gradient Descent**.

Hãy làm theo từng bước hướng dẫn và hoàn thành các phần code được đánh dấu `# YOUR CODE HERE` nhé!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Thiết lập seed để kết quả chạy ngẫu nhiên đồng nhất giữa các lần chạy
np.random.seed(42)
print("Đã import thành công các thư viện cần thiết!")

---
## 1. Hàm Kích Hoạt Sigmoid

Hàm Sigmoid ánh xạ bất kỳ số thực nào vào khoảng $(0, 1)$:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Nhiệm vụ của bạn:** Viết hàm nhận đầu vào `z` (có thể là số thực hoặc một mảng numpy) và trả về giá trị sau khi đi qua hàm Sigmoid.

In [ ]:
def sigmoid(z):
    """
    Tính toán hàm kích hoạt Sigmoid.
    z: Một số thực hoặc một mảng numpy (numpy array).
    """
    # ------------------ YOUR CODE HERE ------------------
    # Thay thế None bằng công thức tính sigmoid
    result = None
    # ----------------------------------------------------
    return result

# --- TEST HÀM SIGMOID ---
z_test = np.array([-10, 0, 10])
sigmoid_test = sigmoid(z_test)
print("Đầu vào:", z_test)
print("Đầu ra mong muốn: Khoảng [0.000045, 0.5, 0.99995]")
print("Đầu ra thực tế của bạn:", sigmoid_test)

# Vẽ đồ thị kiểm tra trực quan
x_vals = np.linspace(-7, 7, 200)
plt.plot(x_vals, sigmoid(x_vals), 'r-', linewidth=2)
plt.grid(True)
plt.title("Đồ thị Hàm kích hoạt Sigmoid")
plt.xlabel("z")
plt.ylabel("sigmoid(z)")
plt.show()

---
## 2. Hàm Mất Mát Binary Cross-Entropy Loss

Hàm mất mát giúp ta đo lường sự sai khác giữa dự đoán của mô hình và nhãn thực tế:
$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

*Lưu ý:* Để tránh lỗi tính toán toán học khi $\hat{y} = 0$ hoặc $\hat{y} = 1$ (khiến cho $\log(0)$ bị lỗi vô cùng), chúng ta thường thêm một đại lượng cực kỳ nhỏ (ví dụ $10^{-15}$) hoặc giới hạn giá trị của dự đoán nằm trong khoảng $[10^{-15}, 1 - 10^{-15}]$. Hãy dùng hàm `np.clip(y_pred, 1e-15, 1 - 1e-15)` trước khi tính log.

**Nhiệm vụ của bạn:** Lập trình hàm tính hàm mất mát Binary Cross-Entropy dưới đây.

In [ ]:
def compute_loss(y_true, y_pred):
    """
    Tính toán hàm mất mát Binary Cross-Entropy.
    y_true: nhãn thực tế (mảng 1 chiều chứa 0 hoặc 1)
    y_pred: xác suất dự đoán bởi mô hình (mảng 1 chiều chứa các giá trị từ 0 đến 1)
    """
    m = len(y_true)
    # Giới hạn y_pred để tránh log(0)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    
    # ------------------ YOUR CODE HERE ------------------
    # Thực hiện tính loss theo công thức
    loss = None
    # ----------------------------------------------------
    return loss

# --- TEST HÀM LOSS ---
y_t = np.array([1, 0, 1, 0])
y_p = np.array([0.9, 0.1, 0.8, 0.2])
loss_val = compute_loss(y_t, y_p)
print(f"Loss thực tế: {loss_val:.6f}")
print("Loss mong muốn: Khoảng ~ 0.16425")

---
## 3. Lập Trình Class Logistic Regression

Giờ là lúc ta ghép nối mọi mảnh ghép lại để tạo nên class `LogisticRegression` hoàn chỉnh.

Đạo hàm (Gradient) của hàm lỗi $J(\theta)$ theo các trọng số:
$$\frac{\partial J(\theta)}{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^{m} \left( \hat{y}^{(i)} - y^{(i)} \right) x_j^{(i)}$$

Dưới dạng ma trận (Vectorization):
$$\text{Gradient} = \frac{1}{m} X^T (\hat{y} - y)$$
$$\text{Bias Gradient} = \frac{1}{m} \sum_{i=1}^m (\hat{y}^{(i)} - y^{(i)})$$

Quy tắc cập nhật trọng số:
$$W := W - \alpha \cdot \text{Gradient}$$
$$b := b - \alpha \cdot \text{Bias Gradient}$$

**Nhiệm vụ của bạn:** Hoàn thành phương thức `fit` để huấn luyện mô hình bằng Gradient Descent và phương thức `predict` để phân loại dữ liệu.

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, epochs=1000):
        """
        lr: Learning rate (tốc độ học)
        epochs: Số lần lặp tối đa qua toàn bộ tập dữ liệu
        """
        self.lr = lr
        self.epochs = epochs
        self.weights = None  # Trọng số W (mảng d dòng, 1 cột hoặc vector 1 chiều)
        self.bias = None     # Trọng số định thiên b (số thực)
        self.loss_history = []
        
    def fit(self, X, y):
        """
        Huấn luyện mô hình từ dữ liệu X và y.
        X: dữ liệu đầu vào, kích thước (m, n) với m là số mẫu, n là số đặc trưng
        y: nhãn đầu ra, kích thước (m,)
        """
        m, n = X.shape
        
        # Khởi tạo trọng số ngẫu nhiên hoặc bằng 0
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.loss_history = []
        
        for epoch in range(self.epochs):
            # BƯỚC 1: Tính đầu ra tuyến tính z và đưa qua hàm sigmoid thu được y_pred
            # z = X * W + b
            # ------------------ YOUR CODE HERE ------------------
            z = None
            y_pred = None
            # ----------------------------------------------------
            
            # BƯỚC 2: Tính toán và lưu trữ Loss để theo dõi
            loss = compute_loss(y, y_pred)
            self.loss_history.append(loss)
            
            # BƯỚC 3: Tính toán Gradient (Độ dốc)
            # dw = (1/m) * X^T * (y_pred - y)
            # db = (1/m) * sum(y_pred - y)
            # ------------------ YOUR CODE HERE ------------------
            dw = None
            db = None
            # ----------------------------------------------------
            
            # BƯỚC 4: Cập nhật trọng số theo Gradient Descent
            # W = W - lr * dw
            # b = b - lr * db
            # ------------------ YOUR CODE HERE ------------------
            self.weights -= None
            self.bias -= None
            # ----------------------------------------------------
            
            # In quá trình học sau mỗi 10% chặng đường
            if epoch % (self.epochs // 10) == 0:
                print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
                
    def predict_proba(self, X):
        """
        Dự đoán xác suất (xác suất mẫu dữ liệu thuộc về lớp 1).
        """
        # ------------------ YOUR CODE HERE ------------------
        # Tính z = X * W + b và chuyển đổi bằng hàm sigmoid
        z = None
        proba = None
        # ----------------------------------------------------
        return proba
        
    def predict(self, X, threshold=0.5):
        """
        Dự đoán lớp (0 hoặc 1) dựa trên ngưỡng (threshold).
        """
        # ------------------ YOUR CODE HERE ------------------
        # Lấy xác suất từ predict_proba(X)
        # Nếu xác suất >= threshold thì phân vào lớp 1, ngược lại phân vào lớp 0
        proba = None
        predictions = None
        # ----------------------------------------------------
        return predictions

---
## 4. Huấn Luyện & Kiểm Thử Trên Dữ Liệu Mô Phỏng

Chúng ta sẽ tạo một tập dữ liệu giả lập phân loại nhị phân bằng thư viện `scikit-learn` để chạy thử mô hình của bạn.

In [ ]:
# 1. Tạo tập dữ liệu giả lập (gồm 1000 mẫu, 2 đặc trưng để dễ vẽ đồ thị)
X_raw, y_raw = make_classification(
    n_samples=1000, 
    n_features=2, 
    n_redundant=0, 
    n_clusters_per_class=1, 
    random_state=42
)

# 2. Phân tách tập dữ liệu thành Train và Test (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# 3. Chuẩn hóa đặc trưng (Feature Scaling) để tăng hiệu suất của Gradient Descent
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Vẽ biểu đồ dữ liệu huấn luyện
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap='bwr', alpha=0.7, edgecolors='k')
plt.title("Dữ liệu huấn luyện sau khi chuẩn hóa (X_train_scaled)")
plt.xlabel("Đặc trưng 1")
plt.ylabel("Đặc trưng 2")
plt.show()

### Bắt đầu chạy huấn luyện mô hình của bạn!

In [ ]:
# Khởi tạo mô hình
model = LogisticRegressionScratch(lr=0.1, epochs=1000)

# Huấn luyện mô hình
print("--- BẮT ĐẦU HUẤN LUYỆN ---")
model.fit(X_train_scaled, y_train)
print("--- HUẤN LUYỆN HOÀN TẤT ---")

print("\nTrọng số học được W:", model.weights)
print("Trọng số định thiên học được b:", model.bias)

### Biểu đồ suy giảm của Hàm Mất Mát (Loss history)

Một mô hình hoạt động tốt sẽ có đường cong mất mát đi xuống mượt mà theo thời gian.

In [ ]:
plt.plot(model.loss_history, 'b-', linewidth=2)
plt.title("Độ giảm thiểu sai số qua từng Epoch")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(True)
plt.show()

---
## 5. Đánh Giá Mô Hình & Vẽ Đường Biên Phân Loại (Decision Boundary)

Hãy đánh giá xem mô hình tự viết của bạn hoạt động tốt như thế nào trên tập dữ liệu kiểm thử (Test Set).

In [ ]:
# Dự đoán trên tập kiểm thử
y_pred = model.predict(X_test_scaled)

# Tính độ chính xác (Accuracy)
accuracy = np.mean(y_pred == y_test)
print(f"Độ chính xác trên tập kiểm thử (Test Set Accuracy): {accuracy * 100:.2f}%")

# Vẽ Đường biên phân loại
x_min, x_max = X_test_scaled[:, 0].min() - 0.5, X_test_scaled[:, 0].max() + 0.5
y_min, y_max = X_test_scaled[:, 1].min() - 0.5, X_test_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# Dự đoán lớp cho mọi điểm trên lưới tọa độ
mesh_points = np.c_[xx.ravel(), yy.ravel()]
Z = model.predict(mesh_points)
Z = Z.reshape(xx.shape)

# Vẽ màu nền đại diện cho vùng phân loại
plt.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
# Vẽ các điểm dữ liệu thực tế kiểm thử
plt.scatter(X_test_scaled[:, 0], X_test_scaled[:, 1], c=y_test, cmap='bwr', edgecolors='k')
plt.title(f"Đường biên Phân loại trên Test Set (Accuracy: {accuracy * 100:.2f}%)")
plt.xlabel("Đặc trưng 1")
plt.ylabel("Đặc trưng 2")
plt.show()

🎉 **Tuyệt vời!** Nếu đồ thị của bạn phân tách chính xác hai cụm màu đỏ và màu xanh và độ chính xác đạt trên **90%**, bạn đã hoàn thành xuất sắc thử thách này rồi!

### 💡 Câu hỏi suy ngẫm dành cho Khang:
1. Nếu tăng tốc độ học `lr` lên quá lớn (ví dụ `lr = 10`), hiện tượng gì sẽ xảy ra với đồ thị Hàm mất mát?
2. Đường biên phân loại của mô hình Logistic Regression này có dạng gì? Tại sao nó không thể phân loại hoàn hảo nếu hai lớp dữ liệu xen lẫn lồng vào nhau theo hình vòng tròn (vấn đề phi tuyến)? Muốn phân loại phi tuyến thì ta cần thay đổi những gì?